### **Semántica vectorial y embeddings** 

Este cuaderno prepara el ingreso a la **semana 2** del curso, centrada en **modelos tempranos de alineamiento visual-semántico, representaciones iniciales y antecedentes del aprendizaje multimodal**.  
El objetivo no es cubrir todos los modelos de lenguaje modernos, sino reforzar la base necesaria para entender:

- la **hipótesis distribucional**,
- la diferencia entre **representaciones dispersas** y **representaciones densas**,
- el papel de la **similitud coseno**,
- la ponderación **TF-IDF**,
- la matriz **término-término** y la ponderación **PPMI**,
- el paso desde semántica vectorial textual hacia un **espacio compartido imagen-texto**.



#### **1. Hipótesis distribucional y representación vectorial**

La idea central es simple: **palabras que ocurren en contextos similares tienden a tener significados similares**.  
Eso permite representar una palabra mediante un vector construido a partir de sus contextos.

##### **Dos familias de representaciones**

- **Dispersas (sparse):** bolsa de palabras, matriz término-documento, matriz de coocurrencias.
- **Densas (dense):** embeddings de baja dimensión aprendidos o inducidos.

Esta base es importante porque los primeros modelos multimodales hacen exactamente esto:

1. obtienen una representación textual,
2. obtienen una representación visual,
3. proyectan ambas a un espacio donde puedan compararse.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
import matplotlib.pyplot as plt


#### **2. Matriz término-documento**

Primero construiremos una representación clásica para un corpus pequeño.  
Esto nos permitirá observar cómo se representan documentos y palabras antes de pasar a embeddings densos.


In [ ]:
corpus = [
    "cat sat on the mat",
    "dog sat on the rug",
    "cat chased mouse",
    "dog chased ball",
    "mouse ate cheese",
    "cat and dog are animals",
]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)
terms = vectorizer.get_feature_names_out()

tdm = pd.DataFrame(X.toarray(), columns=terms, index=[f"doc_{i+1}" for i in range(len(corpus))])
tdm


#### **Interpretación**

- Cada fila es un documento.
- Cada columna es un término del vocabulario.
- Cada celda indica la frecuencia del término en el documento.

Esta representación es útil, pero tiene límites:
- es de alta dimensión,
- es dispersa,
- no capta similitud semántica  entre palabras.


#### **3. Similitud coseno**

La **similitud coseno** mide la cercanía angular entre vectores.  
Es especialmente útil en NLP porque nos interesa más la **dirección** del vector que su magnitud absoluta.

$$
\cos(\theta)=\frac{x \cdot y}{\|x\|\|y\|}
$$

Compararemos algunos documentos del corpus.


In [ ]:
doc_sim = cosine_similarity(X)
pd.DataFrame(doc_sim, index=tdm.index, columns=tdm.index).round(3)


**Lectura rápida**

Observa qué documentos resultan más cercanos:
- los que comparten contexto léxico tienden a mostrar coseno mayor,
- aun así, la representación sigue siendo muy literal: no sabe que *cat* y *dog* son semánticamente similares si no comparten suficiente contexto.


#### **4. TF-IDF: de conteos a ponderación informativa**

Los conteos sin ponderar sobrevaloran palabras frecuentes y poco informativas.  
**TF-IDF** reduce ese problema al equilibrar:
- frecuencia en el documento (**TF**),
- rareza en la colección (**IDF**).

En términos simples: una palabra debe pesar más si es importante en un documento y menos si aparece en todos.


In [ ]:
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(corpus)
terms_tfidf = tfidf.get_feature_names_out()

tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=terms_tfidf, index=tdm.index)
tfidf_df.round(3)


##### **Comparación conceptual**

- **CountVectorizer**: útil para construir la intuición básica.
- **TF-IDF**: mejor cuando queremos distinguir palabras informativas de palabras ubicuas.

Para el curso, esto importa porque la calidad de la representación textual condiciona la calidad del alineamiento multimodal posterior.


#### **5. Matriz término-término y semántica distribucional**

Ahora cambiaremos de perspectiva: en vez de representar documentos por palabras, representaremos **palabras por sus contextos**.

Esto nos acerca más a la lógica de la **semántica distribucional** y a los antecedentes de los embeddings.


In [ ]:
tokenized_docs = [doc.split() for doc in corpus]
vocab = sorted(set(word for doc in tokenized_docs for word in doc))
word_to_idx = {w:i for i,w in enumerate(vocab)}

window_size = 1
cooc = np.zeros((len(vocab), len(vocab)), dtype=np.float64)

for doc in tokenized_docs:
    for i, word in enumerate(doc):
        w_idx = word_to_idx[word]
        start = max(0, i - window_size)
        end = min(len(doc), i + window_size + 1)
        for j in range(start, end):
            if i != j:
                c_idx = word_to_idx[doc[j]]
                cooc[w_idx, c_idx] += 1

cooc_df = pd.DataFrame(cooc, index=vocab, columns=vocab)
cooc_df


##### **Qué representa esta matriz**

- Cada fila es una palabra objetivo.
- Cada columna es una palabra de contexto.
- Cada celda cuenta cuántas veces aparecen cerca.

> **El significado se modela por contexto**, y luego se busca comparar o alinear representaciones.


#### **6. PPMI: una mejor ponderación para coocurrencias**

Cuando construimos una matriz **término-término** o **palabra-contexto**, no siempre conviene trabajar con conteos sin ponderar.  
El problema es que palabras muy frecuentes, como artículos o conectores, tienden a coocurrir con muchas otras simplemente por su frecuencia, no porque exista una relación semántica fuerte.

Para corregir eso, una medida clásica es **PMI** (*Pointwise Mutual Information*), que compara:

- la probabilidad observada de coocurrencia entre una palabra $w$ y un contexto $c$,
- frente a la probabilidad que esperaríamos si fueran **independientes**.

La definición es:

$$
PMI(w,c)=\log_2\frac{P(w,c)}{P(w)P(c)}
$$

donde:

- $P(w,c)$ es la probabilidad conjunta de observar $w$ y $c$ juntos,
- $P(w)$ es la probabilidad marginal de $w$,
- $P(c)$ es la probabilidad marginal de $c$.

##### **Interpretación**
- Si $PMI(w,c) > 0$, entonces $w$ y $c$ coocurren **más de lo esperado por azar**.
- Si $PMI(w,c) = 0$, entonces su coocurrencia es aproximadamente la esperada bajo independencia.
- Si $PMI(w,c) < 0$, entonces coocurren **menos de lo esperado**.

En NLP suele usarse la variante **PPMI** (*Positive PMI*):

$$
PPMI(w,c)=\max(PMI(w,c),0)
$$

##### **¿Por qué usar PPMI en lugar de PMI?**
Hay dos motivos principales:

1. **Los valores negativos suelen ser poco informativos** para representación semántica.  
   En muchos modelos de embeddings distribucionales interesa más reforzar asociaciones fuertes que modelar explícitamente la "no asociación".

2. **PPMI produce matrices más útiles para reducción dimensional**.  
   Al aplicar técnicas como **SVD**, la matriz PPMI suele capturar mejor regularidades semánticas que una matriz de conteos crudos.

##### **Estimación a partir de conteos**
En la práctica, las probabilidades se estiman a partir de una matriz de coocurrencias $M$:

$$
P(w,c)=\frac{M_{wc}}{\sum_{i,j} M_{ij}}
$$

$$
P(w)=\frac{\sum_j M_{wj}}{\sum_{i,j} M_{ij}}
\qquad
P(c)=\frac{\sum_i M_{ic}}{\sum_{i,j} M_{ij}}
$$

Sustituyendo en la definición:

$$
PMI(w,c)=\log_2 \frac{M_{wc}\cdot \sum_{i,j} M_{ij}}{\left(\sum_j M_{wj}\right)\left(\sum_i M_{ic}\right)}
$$

##### **Ventaja conceptual**
PPMI transforma la matriz término-término en una representación que ya no refleja solo frecuencia, sino **fuerza de asociación**.  
Por eso fue una herramienta importante en semántica vectorial antes de los embeddings neuronales modernos.

##### **Limitación importante**
PMI y PPMI tienden a **favorecer eventos raros**: si una coocurrencia aparece pocas veces pero más de lo esperado, puede recibir un valor muy alto.  
Por eso, en aplicaciones reales suele combinarse con:
- umbrales mínimos de frecuencia,
- suavizado,
- o reducción dimensional posterior mediante **SVD**.

##### **Relación con embeddings**
Una idea clásica en semántica distribucional es:

1. construir una matriz palabra-contexto,
2. ponderarla con **PPMI**,
3. aplicar **SVD**,
4. usar las componentes reducidas como embeddings densos.

Esta secuencia conecta directamente con la transición desde representaciones distribucionales clásicas hacia representaciones densas más compactas y comparables.

In [ ]:
total = cooc.sum()
row_sums = cooc.sum(axis=1, keepdims=True)
col_sums = cooc.sum(axis=0, keepdims=True)

ppmi = np.zeros_like(cooc, dtype=float)

for i in range(cooc.shape[0]):
    for j in range(cooc.shape[1]):
        if cooc[i, j] > 0:
            p_wc = cooc[i, j] / total
            p_w = row_sums[i, 0] / total
            p_c = col_sums[0, j] / total
            pmi = np.log2(p_wc / (p_w * p_c))
            ppmi[i, j] = max(pmi, 0)

ppmi_df = pd.DataFrame(ppmi, index=vocab, columns=vocab)
ppmi_df.round(2)


##### **Interpretación**

PPMI suele funcionar mejor que el conteo sin ponderar para captar asociaciones semánticas locales.  
La matriz PPMI refleja la **fuerza de asociación** entre términos.

- Valores altos indican que dos palabras coocurren más de lo esperado por azar.
- Valores cercanos a 0 indican asociación débil o no informativa.
- Esta matriz puede usarse luego como entrada para **reducción dimensional con SVD**, obteniendo embeddings densos.
- 
Históricamente, este tipo de representación fue muy importante antes y junto al auge de Word2Vec.

Además, conecta directamente sobre **semántica distribucional multimodal**, donde también se combinan representaciones textuales y visuales.


#### **7. De matrices dispersas a embeddings densos con SVD**

Un paso clásico en semántica distribucional consiste en transformar una matriz grande y dispersa, por ejemplo, una matriz **PPMI** en una representación **densa** de menor dimensión.

La idea es que la matriz PPMI todavía tiene dos limitaciones importantes:

1. suele ser **muy dispersa**, porque muchas palabras no coocurren directamente,
2. su dimensión crece con el tamaño del vocabulario, lo que la vuelve costosa y poco robusta.

Para resolver esto, se aplica una técnica de **reducción dimensional** como **SVD** (*Singular Value Decomposition*).  
En términos intuitivos, SVD busca identificar las principales direcciones de variación de la matriz y proyectar cada palabra en un espacio más pequeño, donde se conservan las regularidades distribucionales más relevantes.

Si $X$ es la matriz PPMI, la descomposición SVD puede escribirse como:

$$
X \approx U \Sigma V^\top
$$

donde:

- $U$ contiene representaciones asociadas a las palabras,
- $\Sigma$ contiene la importancia de cada componente latente,
- $V^\top$ representa las direcciones principales del espacio.

Al truncar esta descomposición y quedarnos solo con las primeras $k$ componentes, obtenemos embeddings densos de dimensión reducida.

##### **¿Qué gana el modelo al hacer esto?**
- Reduce ruido y redundancia
- Captura asociaciones indirectas entre palabras
- Produce vectores compactos y comparables
- Facilita el uso de medidas como similitud coseno.

##### **Importante**
En este ejemplo usamos `n_components=2` solo con fines didácticos, para inspeccionar el resultado fácilmente.  
En aplicaciones reales, los embeddings suelen tener muchas más dimensiones, por ejemplo 50, 100, 200 o más.

Así, SVD actúa como un puente entre la **semántica distribucional clásica** y las **representaciones densas** que luego dominaron en los modelos neuronales.


In [ ]:
svd = TruncatedSVD(n_components=2, random_state=42)
emb_2d = svd.fit_transform(ppmi)

emb_df = pd.DataFrame(emb_2d, index=vocab, columns=["dim_1", "dim_2"])
emb_df.round(3)


In [ ]:
plt.figure(figsize=(8,6))
for word, (x, y) in emb_df.iterrows():
    plt.scatter(x, y)
    plt.text(x + 0.01, y + 0.01, word, fontsize=10)

plt.title("Embeddings densos inducidos desde PPMI + SVD")
plt.xlabel("dim_1")
plt.ylabel("dim_2")
plt.grid(True, alpha=0.3)
plt.show()


##### **Por qué esto importa**

Este pipeline:**coocurrencias -> ponderación -> reducción de dimensionalidad**— es un antecedente directo de muchos modelos posteriores de representación semántica.

Esto es útil para entender:
- por qué una palabra puede representarse como vector,
- por qué la cercanía vectorial puede interpretarse semánticamente,
- y por qué luego podemos extender la misma lógica a un segundo espacio vectorial, por ejemplo el visual.


#### **8. Similitud entre palabras en el espacio denso**

Ahora evaluaremos similitud coseno entre algunas palabras en el espacio inducido.


In [ ]:
dense_sim = cosine_similarity(emb_2d)
dense_sim_df = pd.DataFrame(dense_sim, index=vocab, columns=vocab)
dense_sim_df.round(3)

#### **9. Embeddings estáticos vs contextuales**

##### **Estáticos**
Modelos como Word2Vec, GloVe o embeddings inducidos desde PPMI asignan **un solo vector por palabra**:
- *mouse* tiene el mismo vector en todos los contextos,
- eso simplifica el modelo, pero no resuelve bien la polisemia.

##### **Contextuales**
Modelos como BERT producen un vector que depende del contexto:
- *mouse* en "the mouse ate cheese" no tiene la misma representación que en "click the mouse".

> Lo importante es saber que los primeros modelos multimodales trabajaban sobre todo con **representaciones textuales estáticas o poco contextualizadas**.


#### **10. De texto a alineamiento multimodal**

En los modelos tempranos de alineamiento visual-semántico se sigue una idea general como esta:

$$
t = E_{text}(x), \qquad v = E_{img}(I)
$$

donde:
- $t$ es el vector textual,
- $v$ es el vector visual.

Luego ambos vectores se proyectan a un espacio compartido:

$$
z_t = W_t t, \qquad z_v = W_v v
$$

y se comparan con una función de similitud, normalmente coseno o producto interno:

$$
s(I,x)=\cos(z_v, z_t) \quad \text{o} \quad z_v^\top z_t
$$

La idea clave es que **sin una buena representación textual no hay alineamiento multimodal útil**.  



#### **11. Preguntas de repaso**

1. Explica por qué la hipótesis distribucional habilita una representación vectorial del significado.
2. Contrasta **matriz término-documento** y **matriz término-término**: ¿qué captura cada una?
3. ¿Por qué la similitud coseno suele preferirse a la distancia euclídea en NLP vectorial?
4. ¿Qué problema intenta corregir **TF-IDF** y por qué **PPMI** es más natural en matrices término-término?
5. ¿Qué se gana y qué se pierde al pasar de matrices dispersas a embeddings densos por reducción de dimensionalidad?
6. ¿Por qué los embeddings estáticos son insuficientes para resolver polisemia?
7. Explica por qué una buena representación textual es condición necesaria para el alineamiento visual-semántico.
8. Describe una posible limitación de usar solo información distribucional textual antes de incorporar visión.


#### **12. Actividad**

1. Cambia el `window_size` de 1 a 2 y compara la matriz de coocurrencias.
2. Prueba con `n_components=3` en SVD y analiza si cambia la estructura del espacio.
3. Selecciona dos pares de palabras que consideres semánticamente cercanas y verifica si el espacio inducido las aproxima.
4. Escribe un párrafo corto respondiendo:  
   **¿Qué le falta a esta representación para convertirse en una representación multimodal?**.


In [ ]:
## Tus respuestas